# 🎛️ Notebook 5 — Gradio Demo
**Project:** Deep Learning Based Arabic Audio Understanding System
**Combines:** Whisper ASR + Emotion Detection + Speaker ID + Keyword Spotting + Summarization
**Hardware:** Kaggle T4 GPU

---
### ⚠️ Prerequisites — download these from Kaggle output panels first:
- `whisper_model.zip`         → from Notebook 1 output
- `best_emotion_model.pth`    → from Notebook 2 output
- `best_speaker_model.pth`    → from Notebook 3 output

Then upload them to this Kaggle session via **Add Data** or place in `/kaggle/working/`.

Run ALL cells top to bottom. Do NOT skip any cell.

## Step 1 — Clean Disk

In [1]:
import os, shutil

print('🧹 Cleaning up disk space...')
dirs_to_clean = ['/kaggle/working/hf_cache']
for path in dirs_to_clean:
    try:
        if os.path.isdir(path): shutil.rmtree(path)
    except: pass

total, used, free = shutil.disk_usage('/kaggle/working')
print(f'💾 Free: {free/1e9:.1f} GB')
print('✅ Ready')

🧹 Cleaning up disk space...
💾 Free: 20.9 GB
✅ Ready


## Step 2 — Install Dependencies

In [2]:
!pip install -q gradio transformers datasets librosa sentence-transformers
!pip install -q torch torchaudio soundfile
print('✅ All packages installed')

✅ All packages installed


## Step 3 — Kaggle Cache Fix

In [3]:
import os

os.makedirs('/kaggle/working/hf_cache', exist_ok=True)
os.makedirs('/kaggle/working/hf_cache/datasets', exist_ok=True)
os.makedirs('/kaggle/working/hf_cache/hub', exist_ok=True)

os.environ['HF_HOME']               = '/kaggle/working/hf_cache'
os.environ['HF_DATASETS_CACHE']     = '/kaggle/working/hf_cache/datasets'
os.environ['TRANSFORMERS_CACHE']    = '/kaggle/working/hf_cache/hub'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/kaggle/working/hf_cache/hub'

print('✅ HuggingFace cache redirected to /kaggle/working/hf_cache')

✅ HuggingFace cache redirected to /kaggle/working/hf_cache


## Step 4 — GPU Check

In [4]:
import torch
import numpy as np
import librosa
import re

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


## Step 5 — Unzip Whisper Model

In [5]:
import os

# Find whisper zip from temp notebook output
WHISPER_ZIP  = None
WHISPER_PATH = '/kaggle/working/whisper-arabic-fleurs-final'

# Search all possible input locations
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f == 'whisper_model.zip':
            WHISPER_ZIP = os.path.join(root, f)

if WHISPER_ZIP and not os.path.exists(WHISPER_PATH):
    print(f'Unzipping {WHISPER_ZIP}...')
    os.makedirs(WHISPER_PATH, exist_ok=True)
    os.system(f'unzip -q {WHISPER_ZIP} -d {WHISPER_PATH}')
    print(f'✅ Unzipped to {WHISPER_PATH}')
elif os.path.exists(WHISPER_PATH):
    print(f'✅ Whisper model already at {WHISPER_PATH}')
else:
    print('⚠️  whisper_model.zip not found')
    print(f'Searched: {[root for root,_,_ in os.walk("/kaggle/input")]}')

Unzipping /kaggle/input/notebooks/nourezz19/zipping-file/whisper_model.zip...
✅ Unzipped to /kaggle/working/whisper-arabic-fleurs-final


## Step 6 — Load Whisper + Summarization + Sentence Encoder

In [6]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer

# ── 1. Whisper ────────────────────────────────────────────────────────────────
print('Loading Whisper...')
try:
    whisper_processor = WhisperProcessor.from_pretrained(WHISPER_PATH, local_files_only=True)
    whisper_model     = WhisperForConditionalGeneration.from_pretrained(WHISPER_PATH, local_files_only=True).to(device)
    whisper_model.eval()
    print('  ✅ Whisper loaded')
except Exception as e:
    print(f'  ⚠️  {e} — loading fallback whisper-small')
    whisper_processor = WhisperProcessor.from_pretrained('openai/whisper-small')
    whisper_model     = WhisperForConditionalGeneration.from_pretrained('openai/whisper-small').to(device)
    whisper_model.eval()
    print('  ✅ Fallback whisper-small loaded')

# ── 2. Summarization on CPU ───────────────────────────────────────────────────
print('Loading mT5 summarization (CPU)...')
SUMM_MODEL     = 'csebuetnlp/mT5_multilingual_XLSum'
tokenizer_summ = AutoTokenizer.from_pretrained(SUMM_MODEL)
model_summ     = AutoModelForSeq2SeqLM.from_pretrained(SUMM_MODEL).to('cpu')
model_summ.eval()
print('  ✅ mT5 loaded on CPU')

# ── 3. Sentence encoder on CPU ────────────────────────────────────────────────
print('Loading sentence encoder (CPU)...')
encoder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device='cpu')
print('  ✅ Sentence encoder loaded')

print('\n✅ All base models loaded')

Loading Whisper...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

  ✅ Whisper loaded
Loading mT5 summarization (CPU)...


config.json:   0%|          | 0.00/730 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  ✅ Sentence encoder loaded

✅ All base models loaded


## Step 7 — Load Emotion + Speaker Models

In [7]:
import torch.nn as nn
import torch.nn.functional as F

# ── Find model files ──────────────────────────────────────────────────────────
EMOTION_PATH = None
SPEAKER_PATH = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f == 'best_emotion_model.pth':
            EMOTION_PATH = os.path.join(root, f)
        if f == 'best_speaker_model.pth':
            SPEAKER_PATH = os.path.join(root, f)

print(f'Emotion model: {EMOTION_PATH}')
print(f'Speaker model: {SPEAKER_PATH}')

# ── Emotion Model ─────────────────────────────────────────────────────────────
class CNN_BiLSTM_Emotion(nn.Module):
    def __init__(self, n_classes=4):
        super().__init__()
        self.cnn1 = nn.Sequential(
            nn.Conv2d(1, 32, (3,3), padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, (3,3), padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d((2,2)), nn.Dropout2d(0.25),
        )
        self.cnn2 = nn.Sequential(
            nn.Conv2d(32, 64, (3,3), padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, (3,3), padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d((2,2)), nn.Dropout2d(0.25),
        )
        self.cnn3 = nn.Sequential(
            nn.Conv2d(64, 128, (3,3), padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d((2,2)), nn.Dropout2d(0.25),
        )
        self.bilstm = nn.LSTM(128*15, 256, num_layers=2, batch_first=True,
                               bidirectional=True, dropout=0.3)
        self.classifier = nn.Sequential(
            nn.Linear(512, 128), nn.ReLU(), nn.Dropout(0.4), nn.Linear(128, n_classes)
        )
    def forward(self, x):
        x = self.cnn1(x); x = self.cnn2(x); x = self.cnn3(x)
        B, C, H, W = x.shape
        x = x.permute(0, 3, 1, 2).reshape(B, W, C*H)
        x, _ = self.bilstm(x)
        return self.classifier(x[:, -1, :])

IDX2EMOTION = {0: 'neutral 😐', 1: 'happy 😊', 2: 'sad 😢', 3: 'angry 😡'}

emotion_model = None
try:
    emotion_model = CNN_BiLSTM_Emotion().to(device)
    emotion_model.load_state_dict(
        torch.load(EMOTION_PATH, map_location=device)
    )
    emotion_model.eval()
    print('✅ Emotion model loaded')
except Exception as e:
    print(f'⚠️  Emotion model error: {e}')

# ── Speaker Model ─────────────────────────────────────────────────────────────
class SpeakerEncoder(nn.Module):
    def __init__(self, n_speakers=2, embed_dim=256):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, (3,3), padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, (3,3), padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d((2,2)),
            nn.Conv2d(32, 64, (3,3), padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, (3,3), padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d((2,2)),
            nn.Conv2d(64, 128, (3,3), padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d((2,2)),
        )
        self.pool_input_size = 128 * 5
        self.embed_layers = nn.Sequential(
            nn.Linear(self.pool_input_size * 2, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, embed_dim), nn.ReLU(),
        )
        self.classifier = nn.Linear(embed_dim, n_speakers)

    def forward(self, x, return_embedding=False):
        x = self.cnn(x)
        B, C, H, T = x.shape
        x = x.reshape(B, C*H, T)
        x = torch.cat([x.mean(dim=2), x.std(dim=2)], dim=1)
        emb = self.embed_layers(x)
        if return_embedding:
            return emb
        return self.classifier(emb)

speaker_model = None
try:
    ckpt  = torch.load(SPEAKER_PATH, map_location=device)
    n_spk = ckpt['classifier.weight'].shape[0]
    speaker_model = SpeakerEncoder(n_speakers=n_spk).to(device)
    speaker_model.load_state_dict(ckpt)
    speaker_model.eval()
    print(f'✅ Speaker model loaded ({n_spk} speakers)')
except Exception as e:
    print(f'⚠️  Speaker model error: {e}')

print('\n✅ All models ready')

Emotion model: /kaggle/input/datasets/nourezz19/best-models/best_emotion_model.pth
Speaker model: /kaggle/input/datasets/nourezz19/best-models/best_speaker_model.pth
✅ Emotion model loaded
✅ Speaker model loaded (2 speakers)

✅ All models ready


## Step 8 — Define Processing Functions

In [8]:
SR_WHISPER = 16000
N_MFCC     = 40
MAX_LEN    = 200
N_FILTERS  = 40
MAX_FBANK  = 300

KEYWORD_GROUPS = {
    '🔴 Emergency': ['طوارئ', 'نجدة', 'استغاثة', 'خطر', 'حادث'],
    '🟡 Deadline':  ['موعد', 'مهلة', 'الموعد النهائي', 'تسليم'],
    '🟠 Exam':      ['امتحان', 'اختبار', 'مذاكرة', 'درجة'],
    '🔵 Meeting':   ['اجتماع', 'مقابلة', 'ندوة', 'مؤتمر'],
    '🟣 Important': ['مهم', 'ضروري', 'عاجل'],
}

SEMANTIC_ANCHORS = {
    'emergency': ['طوارئ', 'نجدة', 'خطر شديد'],
    'exam':      ['امتحان', 'اختبار الطلاب'],
    'deadline':  ['موعد التسليم', 'مهلة نهائية'],
    'meeting':   ['اجتماع', 'لقاء عمل'],
}

anchor_embeddings = {}
for category, phrases in SEMANTIC_ANCHORS.items():
    embs = encoder.encode(phrases, normalize_embeddings=True)
    anchor_embeddings[category] = embs.mean(axis=0)

def normalize_ar(text):
    text = re.sub(r'[\u0610-\u061A\u064B-\u065F]', '', text)
    text = re.sub(r'[أإآ]', 'ا', text)
    return text.strip()

def transcribe_audio(audio_path: str) -> str:
    audio, _ = librosa.load(audio_path, sr=SR_WHISPER)
    inputs = whisper_processor(
        audio, sampling_rate=SR_WHISPER, return_tensors='pt'
    ).input_features.to(device)
    with torch.no_grad():
        ids = whisper_model.generate(inputs, language='ar')
    return whisper_processor.batch_decode(ids, skip_special_tokens=True)[0]

def detect_emotion(audio_path: str) -> str:
    if emotion_model is None:
        return '⚠️ Emotion model not loaded'
    audio, _ = librosa.load(audio_path, sr=22050)
    mfcc   = librosa.feature.mfcc(y=audio, sr=22050, n_mfcc=N_MFCC)
    delta  = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    feat   = np.vstack([mfcc, delta, delta2])
    feat   = (feat - feat.mean(1, keepdims=True)) / (feat.std(1, keepdims=True) + 1e-8)
    if feat.shape[1] < MAX_LEN:
        feat = np.pad(feat, ((0,0), (0, MAX_LEN - feat.shape[1])))
    else:
        feat = feat[:, :MAX_LEN]
    x = torch.FloatTensor(feat).unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = F.softmax(emotion_model(x), dim=1)[0]
    result = ''
    for label, prob in zip(IDX2EMOTION.values(), probs):
        bar = '█' * int(prob.item() * 15)
        result += f'{label}: {bar:<15} {prob.item()*100:.1f}%\n'
    return result

def identify_speaker(audio_path: str) -> str:
    if speaker_model is None:
        return '⚠️ Speaker model not loaded'
    audio, _ = librosa.load(audio_path, sr=SR_WHISPER)
    mel = librosa.feature.melspectrogram(
        y=audio, sr=SR_WHISPER, n_mels=N_FILTERS,
        hop_length=160, win_length=400, fmin=20, fmax=7600
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)
    log_mel = (log_mel - log_mel.mean()) / (log_mel.std() + 1e-8)
    if log_mel.shape[1] < MAX_FBANK:
        log_mel = np.pad(log_mel, ((0,0), (0, MAX_FBANK - log_mel.shape[1])))
    else:
        log_mel = log_mel[:, :MAX_FBANK]
    x = torch.FloatTensor(log_mel).unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        emb    = speaker_model(x, return_embedding=True)
        logits = speaker_model.classifier(emb)
        probs  = F.softmax(logits, dim=1)[0]
    labels = ['male 👨', 'female 👩']
    result = 'Gender Detection:\n'
    for i, (prob, label) in enumerate(zip(probs, labels)):
        bar = '█' * int(prob.item() * 15)
        result += f'  {label}: {bar:<15} {prob.item()*100:.1f}%\n'
    return result

def spot_keywords_text(transcript: str) -> str:
    norm  = normalize_ar(transcript)
    found = []
    for category, keywords in KEYWORD_GROUPS.items():
        for kw in keywords:
            if normalize_ar(kw) in norm:
                found.append(f'{category}: "{kw}"')
    sent_emb = encoder.encode([transcript], normalize_embeddings=True)[0]
    for category, anchor_emb in anchor_embeddings.items():
        score = float(np.dot(sent_emb, anchor_emb))
        if score > 0.45:
            found.append(f'🔷 Semantic [{category}]: score={score:.2f}')
    return '\n'.join(found) if found else 'No target keywords detected.'

def summarize_text(transcript: str) -> str:
    if len(transcript.strip()) < 50:
        return '(Text too short to summarize)'
    inputs = tokenizer_summ(
        transcript.strip(), return_tensors='pt',
        padding='max_length', truncation=True, max_length=512
    ).to('cpu')  # CPU only — avoids CUDA kernel issue
    with torch.no_grad():
        ids = model_summ.generate(
            inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            num_beams=4, max_new_tokens=80, early_stopping=True
        )
    return tokenizer_summ.decode(ids[0], skip_special_tokens=True,
                                  clean_up_tokenization_spaces=False)

print('✅ All processing functions ready')

✅ All processing functions ready


## Step 9 — Build and Launch Gradio Interface

In [9]:
import gradio as gr

def process_audio(audio_path, do_transcribe, do_emotion, do_speaker, do_keywords, do_summary):
    if audio_path is None:
        return 'No audio provided ⚠️', '', '', '', ''

    transcript = ''
    if do_transcribe or do_keywords or do_summary:
        try:
            transcript = transcribe_audio(audio_path)
        except Exception as e:
            transcript = f'Error: {e}'

    emotion_result = ''
    if do_emotion:
        try: emotion_result = detect_emotion(audio_path)
        except Exception as e: emotion_result = f'Error: {e}'

    speaker_result = ''
    if do_speaker:
        try: speaker_result = identify_speaker(audio_path)
        except Exception as e: speaker_result = f'Error: {e}'

    keyword_result = ''
    if do_keywords and transcript:
        try: keyword_result = spot_keywords_text(transcript)
        except Exception as e: keyword_result = f'Error: {e}'

    summary_result = ''
    if do_summary and transcript:
        try: summary_result = summarize_text(transcript)
        except Exception as e: summary_result = f'Error: {e}'

    return transcript, emotion_result, speaker_result, keyword_result, summary_result


with gr.Blocks(title='Arabic Audio Intelligence System', theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🎙️ Arabic Audio Intelligence System
    **Deep Learning-based Arabic Speech Understanding**
    Upload or record Arabic audio to get transcript, emotion, speaker ID, keywords, and summary.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown('### 🎤 Input')
            audio_input = gr.Audio(
                sources=['microphone', 'upload'],
                type='filepath',
                label='Arabic Audio (.wav / .mp3)'
            )
            gr.Markdown('### ⚙️ Select Modules')
            do_transcribe = gr.Checkbox(value=True,  label='🎙️ Speech-to-Text (Whisper)')
            do_emotion    = gr.Checkbox(value=True,  label='😊 Emotion Detection (CNN-BiLSTM)')
            do_speaker    = gr.Checkbox(value=True,  label='👤 Speaker Identification')
            do_keywords   = gr.Checkbox(value=True,  label='🔍 Keyword Spotting')
            do_summary    = gr.Checkbox(value=False, label='📝 Summarization (mT5) — slower')
            analyze_btn   = gr.Button('🚀 Analyze Audio', variant='primary', size='lg')

        with gr.Column(scale=2):
            gr.Markdown('### 📊 Results')
            with gr.Tab('🎙️ Transcript'):
                transcript_out = gr.Textbox(label='Arabic Transcript',
                                             lines=5, rtl=True, show_copy_button=True)
            with gr.Tab('😊 Emotion'):
                emotion_out = gr.Textbox(label='Detected Emotion', lines=6)
            with gr.Tab('👤 Speaker'):
                speaker_out = gr.Textbox(label='Speaker Identification', lines=5)
            with gr.Tab('🔍 Keywords'):
                keyword_out = gr.Textbox(label='Detected Keywords', lines=6)
            with gr.Tab('📝 Summary'):
                summary_out = gr.Textbox(label='Arabic Summary',
                                          lines=4, rtl=True, show_copy_button=True)

    gr.Markdown("""
    ---
    **Models:** Whisper-small (fine-tuned FLEURS ar_eg) • CNN-BiLSTM (RAVDESS) • x-vector CNN (FLEURS) • mT5 XL-Sum
    **Tips:** Speak clearly in Arabic for 5–15 seconds. Try keywords: امتحان، موعد، اجتماع، طوارئ
    """)

    analyze_btn.click(
        fn=process_audio,
        inputs=[audio_input, do_transcribe, do_emotion, do_speaker, do_keywords, do_summary],
        outputs=[transcript_out, emotion_out, speaker_out, keyword_out, summary_out]
    )

# ✅ Works on Kaggle (share=True → public URL) and VS Code (localhost:7860)
print('🚀 Launching Gradio demo...')
demo.launch(share=True, quiet=True)
print('✅ Copy the public URL above and open it in your browser')

/tmp/ipykernel_57/2294826736.py:37: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title='Arabic Audio Intelligence System', theme=gr.themes.Soft()) as demo:


🚀 Launching Gradio demo...
* Running on public URL: https://ef911ac1b149d03fc7.gradio.live


✅ Copy the public URL above and open it in your browser


## How to Run Locally in VS Code (after downloading from Kaggle)

### 1. Download from Kaggle Output panels
```
Notebook 1 output → whisper_model.zip
Notebook 2 output → best_emotion_model.pth
Notebook 3 output → best_speaker_model.pth
```

### 2. Folder structure on your PC
```
my_project/
├── 05_gradio_demo.ipynb
├── best_emotion_model.pth
├── best_speaker_model.pth
└── whisper-arabic-fleurs-final/   ← unzip whisper_model.zip here
    ├── config.json
    ├── model.safetensors
    └── ...
```

### 3. Install locally
```bash
pip install gradio transformers datasets librosa sentence-transformers torch torchaudio soundfile
```

### 4. Change paths and launch
In Step 5 cell, change:
```python
WHISPER_ZIP  = './whisper_model.zip'
WHISPER_PATH = './whisper-arabic-fleurs-final'
```
In Step 7 cell, change:
```python
torch.load('./best_emotion_model.pth', ...)
torch.load('./best_speaker_model.pth', ...)
```
In Step 9 cell (last line), change:
```python
demo.launch(share=False, server_name='0.0.0.0', server_port=7860)
```
Then open **http://localhost:7860** in your browser.

### 5. No GPU locally?
```python
device = 'cpu'  # add this at the top of Step 4
```
Inference takes ~5-10 sec per clip but works fine.